In [1]:
#Importing required libraries
import matplotlib.pyplot as plt
import random
import copy

In [2]:
#Game State definition:
state = {
    "player_1": [],
    "player_2": [],
    "player_3": [],
    "top_card": [],
    "turn": [],
    "deck": []
}

In [3]:
#Card class definition
class Card:
    def __init__(self, color, value):

        self.color = color
        self.value = value

    def __repr__(self):

        return f"{self.color} {self.value}"

    def __eq__(self, other):

        if not isinstance(other, Card):
            return False

        return self.color == other.color and self.value == other.value

    def __hash__(self):

        return hash((self.color, self.value))

In [4]:
#Generating Deck Of Cards
def generate_deck():

  #Defining colors
  colors = ["Red", "Blue", "Green", "Yellow"]
  deck = []

  #Every number + Skip for each color
  for color in colors:
    for i in range(1, 10):
      deck.append(Card(color, i))
    deck.append(Card(color, "Skip")) #Adding Skip

  #Shuffling using random
  random.shuffle(deck)

  return deck

In [5]:
#Dealing Cards as list, 5 each:
def deal_cards(deck):

  return [deck.pop() for _ in range(5)]

In [6]:
# Terminal State / game over (one player wins)
def is_terminal(state):

  return (len(state["p1"]) == 0 or len(state["p2"]) == 0 or len(state["p3"]) == 0)

In [7]:
#Evaluation Function: Score = 50 − 5(CAI) + 2(Copp) + 3(S)
def evaluate(state, player):

  #CAI
  my_cards = len(state[player])

  #Copp
  oppenents = [x for x in ["p1", "p2", "p3"] if x != player]
  opp_cards = 0
  for opp in oppenents:
    opp_cards += len(state[opp])

  #S
  skip_cards = sum(1 for c in state[player] if c.value == "Skip")

  score = 50 - 5 * my_cards + 2 * opp_cards + 3 * skip_cards

  return score


In [8]:
#Valid Move Check (Returns list of valid moves)
def valid_moves(hand, top_card):

  valid = []

  #Appending valid moves
  for card in hand:

    if card.color == top_card.color or card.value == top_card.value: #Validation Check
      valid.append(card)

  #If no valid moves then draw from deck
  if not valid:

    return ["DRAW"]

  return valid

In [9]:
#Evaluation Function: Score = 50 − 5(CAI) + 2(Copp) + 3(S)
def evaluate(state, player):

  #CAI
  my_cards = len(state[player])

  #Copp
  oppenents = [x for x in ["p1", "p2", "p3"] if x != player]
  opp_cards = 0
  for opp in oppenents:
    opp_cards += len(state[opp])

  #S
  skip_cards = sum(1 for c in state[player] if c.value == "Skip")

  score = 50 - 5 * my_cards + 2 * opp_cards + 3 * skip_cards

  return score

In [10]:
# State Transition : apply move function:
def apply_move(state, player, move):

  new_state = copy.deepcopy(state)

  #Handling Draw:
  if move == "DRAW":
    new_state[player].append(new_state["deck"].pop())

    #Next player's turn
    new_state["turn"] = (new_state["turn"] + 1) % 3
    return new_state

  #DEBUG
  if move not in new_state[player]:
    print("ERROR: Move not in hand!")
    print("Move:", move)
    print("Hand:", new_state[player])
    raise Exception("Invalid move")

  #Playing Card
  new_state[player].remove(move)
  new_state["top_card"] = move

  #Handling Skip:
  if move.value == "Skip":
    new_state["turn"] = (new_state["turn"] + 2) % 3
  else:
    #Normal Next Turn:
    new_state["turn"] = (new_state["turn"] + 1) % 3

  return new_state